In [0]:
import requests
import pandas as pd
from datetime import datetime

In [0]:
def extrair_dados_bitcoin():
  url = ("https://api.coinbase.com/v2/prices/spot")
  resposta = requests.get(url)
  return resposta.json()  


In [0]:
def extrair_cotacao_usd_brl():
    api_key = 'api_key'
    url = f'https://api.currencyfreaks.com/v2.0/rates/latest?apikey={api_key}'
    resposta = requests.get(url)
    return resposta.json()

In [0]:
def tratar_dados_bitcoin(dados_json, taxa_usd_brl):
    valor_usd = float(dados_json['data']['amount'])
    criptomoeda = dados_json['data']['base']
    moeda_original = dados_json['data']['currency']

    valor_brl = valor_usd * taxa_usd_brl

    timestamp = datetime.now()

    dados_tratados = [{
        "valor_usd": valor_usd,
        "valor_brl": valor_brl,
        "criptomoeda": criptomoeda,
        "moeda_original": moeda_original,
        "taxa_conversao_usd_brl": taxa_usd_brl,
        "timestamp": timestamp
    }]

    return dados_tratados

In [0]:
Dados_bitcoin = extrair_dados_bitcoin()
dados_cotacao = extrair_cotacao_usd_brl()

taxa_usd_brl = float(dados_cotacao['rates']['BRL'])

dados_bitcoin_tratado = tratar_dados_bitcoin(Dados_bitcoin, taxa_usd_brl)

print(dados_bitcoin_tratado)

# Configurando Unity Catalog

In [0]:
%sql
CREATE CATALOG IF NOT EXISTS pipeline_api_bitcoin
COMMENT 'Catálogo de demonstração criado para ao nosso projeto bitcoin';

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS pipeline_api_bitcoin.lakehouse
COMMENT 'Schema lakehouse para salvar dados processados';

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS pipeline_api_bitcoin.bitcoin_data
COMMENT 'Schema lakehouse para salvar dados processados';

In [0]:
%sql
CREATE VOLUME IF NOT EXISTS pipeline_api_bitcoin.lakehouse.raw_files
COMMENT 'Volume para arquivos brutos de ingestão inicial';

# Criando Pandas DataFrame

In [0]:
df_pandas = pd.DataFrame(dados_bitcoin_tratado)

print(df_pandas)

# Salvando em JSON usando Pandas

In [0]:
event_ts = dados_bitcoin_tratado[0]["timestamp"]

ts = event_ts.strftime("%Y%m%d%H%M%S_%f")

json_file = f"/Volumes/pipeline_api_bitcoin/lakehouse/raw_files/bitcoin_{ts}.json"

df_pandas.to_json(json_file, orient='records', date_format='iso', indent=2)
print(f"Arquivo JSON salvo: {json_file}")

# Salvando em CSV usando Pandas

In [0]:
csv_file = f"/Volumes/pipeline_api_bitcoin/lakehouse/raw_files/bitcoin_{ts}.csv"

df_pandas.to_csv(csv_file, index=False)
print(f"Arquivo CSV salvo: {csv_file}")



# Salvando em Parquet usando Pandas

In [0]:
parquet_file = f"/Volumes/pipeline_api_bitcoin/lakehouse/raw_files/bitcoin_{ts}.parquet"

df_pandas.to_parquet(parquet_file)
print(f"Arquivo Parquet salvo: {parquet_file}")


# Salvando como Delta Table

In [0]:
df_spark = spark.createDataFrame(df_pandas)
df_spark.printSchema()

In [0]:
df_spark.write.format('delta').mode('append').saveAsTable('pipeline_api_bitcoin.bitcoin_data.bitcoin_data')

In [0]:
%sql
SELECT * FROM pipeline_api_bitcoin.bitcoin_data.bitcoin_data

# Verificando Histórico da Delta Table (Time Travel)

In [0]:
%sql
DESCRIBE HISTORY pipeline_api_bitcoin.bitcoin_data.bitcoin_data